# CircuitLM — 10K Vocab + Large Corrector Training (Kaggle GPU)

Trains a **hybrid CircuitLM** (integer PDA circuit + neural corrector) with **10,000-token BPE vocab** on personal conversations.

**Corrector sizes trained:** Tiny · Small · Medium · Large

**Download all 4 after training.**

**Data:** `/kaggle/input/datasets/zacharymaronek/circuit-lm-personal/all_personal_training.txt`

**Runtime:** GPU required. P100 or T4. ~30–60 min total.

## 1. Clone CircuitLM + Install Dependencies

In [ ]:
# Clone the circuit_lm repo
!git clone https://github.com/toxzak-svg/circuit_lm.git /tmp/circuit_lm

# Install dependencies
%pip install ortools msgpack sentencepiece torch --quiet

import sys
sys.path.insert(0, '/tmp/circuit_lm/src')
print('Setup complete')

## 2. Load Training Data

In [ ]:
import os

DATA_PATH = '/kaggle/input/datasets/zacharymaronek/circuit-lm-personal/all_personal_training.txt'

with open(DATA_PATH, encoding='utf-8', errors='replace') as f:
    text = f.read()

lines = [l for l in text.strip().split('\n') if l.strip()]
print(f'Loaded {len(lines)} lines ({len(text)/1024/1024:.1f} MB)')

## 3. Tokenizer — 10K BPE Vocab

Large vocab captures more personal style details.

In [ ]:
import time
from circuit_lm.tokenizer import Tokenizer
from circuit_lm.data import load_sequences

VOCAB_SIZE = 10000

print(f'Building {VOCAB_SIZE}-token BPE tokenizer...')
t0 = time.time()
tokenizer = Tokenizer.from_text(
    text,
    vocab_size=VOCAB_SIZE,
    mode='bpe',
    bpe_merges=VOCAB_SIZE,
)
print(f'  vocab={tokenizer.vocab_size}, took {time.time()-t0:.1f}s')

print('Tokenizing...')
t0 = time.time()
sequences = load_sequences(text, tokenizer)
total_tokens = sum(len(s) for s in sequences)
print(f'  {len(sequences)} sequences, {total_tokens:,} tokens, took {time.time()-t0:.1f}s')
print(f'  Compression: {len(text)/total_tokens:.1f} chars/token')

## 4. Train PDA Circuit (OR-Tools CP-SAT)

Integer-only PDA with stack. Zero floats — pure integer arithmetic at inference.

In [ ]:
from circuit_lm.train_joint_pda_cpsat import train_joint_pda
import time

# Circuit hyperparameters — keep circuit small so it generalizes
STATE_BITS = 6       # 64 states
STACK_DEPTH = 4      # PDA stack depth
STACK_STEPS = 15     # Stack operation budget
TRANS_STEPS = 22     # Transition computation budget
EMIT_STEPS = 23      # Emission computation budget

print(f'Training PDA circuit: {STATE_BITS} bits, stack_depth={STACK_DEPTH}')
t0 = time.time()
circuit = train_joint_pda(
    sequences=sequences,
    vocab_size=tokenizer.vocab_size,
    state_bits=STATE_BITS,
    stack_depth=STACK_DEPTH,
    stack_steps=STACK_STEPS,
    transition_steps=TRANS_STEPS,
    emission_steps=EMIT_STEPS,
)
circuit_time = time.time() - t0
print(f'Circuit trained in {circuit_time:.1f}s')

## 5. Save Circuit + Tokenizer (reused for all correctors)

In [ ]:
from circuit_lm.io import save_model

CIRCUIT_PATH = '/tmp/circuit_pda_10k.json'
save_model(circuit, tokenizer, CIRCUIT_PATH)
print(f'Circuit saved: {os.path.getsize(CIRCUIT_PATH)/1024:.0f} KB')

## 6. Train 4 Corrector Sizes

All 4 share the same circuit. Only the neural corrector changes.

| Size    | embed_dim | hidden_dim | num_layers | Params (approx) |
|---------|-----------|------------|------------|-----------------|
| Tiny    | 64        | 128        | 2          | ~1.5 M          |
| Small   | 128       | 256        | 3          | ~5 M            |
| Medium  | 256       | 512        | 4          | ~20 M           |
| Large   | 512       | 1024       | 4          | ~80 M           |

P100 (16GB) can handle Medium comfortably. Use T4 if you only have 15GB.

In [ ]:
import torch
import time
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# ---- Corrector Configs ----
CORRECTOR_CONFIGS = {
    'tiny':   dict(embed_dim=64,   hidden_dim=128,  num_layers=2, max_context_len=32),
    'small':  dict(embed_dim=128,  hidden_dim=256,  num_layers=3, max_context_len=64),
    'medium': dict(embed_dim=256,  hidden_dim=512,  num_layers=4, max_context_len=64),
    'large':  dict(embed_dim=512,  hidden_dim=1024, num_layers=4, max_context_len=128),
}

# Training budgets per corrector (tiny=fast, large=slow)
TRAINING_BUDGETS = {
    'tiny':   dict(num_epochs=5,  batch_size=256, max_examples=80000),
    'small':  dict(num_epochs=5,  batch_size=128, max_examples=80000),
    'medium': dict(num_epochs=5,  batch_size=64,  max_examples=60000),
    'large':  dict(num_epochs=3,  batch_size=32,  max_examples=40000),
}

# Which sizes to actually train (adjust based on GPU time budget)
# Set to ['tiny','small','medium','large'] to train all 4
SIZES_TO_TRAIN = ['tiny', 'small', 'medium', 'large']

print('Corrector sizes to train:', SIZES_TO_TRAIN)

In [ ]:
from hybrid import train_hybrid, HybridModel, NeuralCorrector
from circuit_lm.io import load_model

RESULTS = {}

for size in SIZES_TO_TRAIN:
    print('\n' + '='*50)
    print(f'TRAINING CORRECTOR: {size.upper()}')
    print('='*50)
    
    cfg = CORRECTOR_CONFIGS[size]
    tcfg = TRAINING_BUDGETS[size]
    
    print(f'  embed_dim={cfg["embed_dim"]}, hidden_dim={cfg["hidden_dim"]}, '
          f'num_layers={cfg["num_layers"]}, context_len={cfg["max_context_len"]}')
    print(f'  epochs={tcfg["num_epochs"]}, batch_size={tcfg["batch_size"]}, '
          f'max_examples={tcfg["max_examples"]}')
    
    # Adjust batch_size down if we hit OOM
    batch_size = tcfg['batch_size']
    
    t0 = time.time()
    try:
        corrector = train_hybrid(
            circuit_path=CIRCUIT_PATH,
            data_path=DATA_PATH,
            output_path=f'/tmp/corrector_{size}_10k.pt',
            num_epochs=tcfg['num_epochs'],
            batch_size=batch_size,
            circuit_weight=0.5,
            max_examples=tcfg['max_examples'],
            # Pass corrector architecture overrides
            embed_dim=cfg['embed_dim'],
            hidden_dim=cfg['hidden_dim'],
            num_layers=cfg['num_layers'],
            max_context_len=cfg['max_context_len'],
        )
        elapsed = time.time() - t0
        
        # Count params
        num_params = sum(p.numel() for p in corrector.parameters())
        
        # Quick eval
        hybrid, tok = HybridModel.load(CIRCUIT_PATH, f'/tmp/corrector_{size}_10k.pt')
        hybrid.corrector.to(device)
        
        # Sample test
        prompt_ids = tok.encode('The best way to build AI that you can trust is')
        from hybrid import generate_reply_hybrid
        reply_ids = generate_reply_hybrid(hybrid, tok, prompt_ids, max_tokens=40)
        reply = tok.decode(reply_ids)
        
        RESULTS[size] = {
            'status': 'success',
            'time': elapsed,
            'params': num_params,
            'sample': reply,
            'config': {**cfg, **tcfg},
        }
        
        print(f'  ✓ {size} done in {elapsed:.1f}s — {num_params:,} params')
        print(f'  Sample: {reply[:120]}...')
        
        # Clear GPU memory between sizes
        del corrector, hybrid
        torch.cuda.empty_cache()
        
    except RuntimeError as e:
        if 'out of memory' in str(e):
            print(f'  ✗ OOM on {size} — try reducing batch_size or max_examples')
            RESULTS[size] = {'status': 'OOM', 'config': {**cfg, **tcfg}}
            torch.cuda.empty_cache()
        else:
            raise

## 7. Results Summary

In [ ]:
print('\n' + '='*60)
print('TRAINING RESULTS')
print('='*60)
print(f'Vocabulary: {VOCAB_SIZE} tokens')
print(f'Circuit: PDA, {STATE_BITS} bits, stack_depth={STACK_DEPTH}')
print()

for size in SIZES_TO_TRAIN:
    r = RESULTS.get(size, {})
    status = r.get('status', 'not_run')
    if status == 'success':
        cfg = r['config']
        print(f'{size.upper():8} | {r["params"]:>8,} params | {r["time"]:>6.1f}s | '
              f'embed={cfg["embed_dim"]:4d} hidden={cfg["hidden_dim"]:4d} layers={cfg["num_layers"]}')
    elif status == 'OOM':
        print(f'{size.upper():8} | OOM — reduce batch_size or max_examples')
    else:
        print(f'{size.upper():8} | {status}')

## 8. Download All Files

In [ ]:
from IPython.display import FileLink, display

print('='*60)
print('DOWNLOAD YOUR TRAINED FILES')
print('='*60)
print()

# Always save circuit + tokenizer info (they're shared)
print('CIRCUIT (shared by all corrector sizes):')
display(FileLink(CIRCUIT_PATH))
print()

print('CORRECTOR MODELS:')
for size in SIZES_TO_TRAIN:
    if RESULTS.get(size, {}).get('status') == 'success':
        path = f'/tmp/corrector_{size}_10k.pt'
        r = RESULTS[size]
        print(f'  [{size.upper()}] {r["params"]:,} params, {r["time"]:.1f}s —', end=' ')
        display(FileLink(path))
        print()

# Save tokenizer info
tokenizer_info = {
    'vocab_size': tokenizer.vocab_size,
    'mode': 'bpe',
    'circuit_vocab_size': tokenizer.vocab_size,
}
with open('/tmp/tokenizer_info.json', 'w') as f:
    json.dump(tokenizer_info, f)
print('Tokenizer config:')
display(FileLink('/tmp/tokenizer_info.json'))

print()
total_time = sum(r.get('time', 0) for r in RESULTS.values() if r.get('status') == 'success')
print(f'Total GPU training time: {total_time:.1f}s')

## 9. Generate Test — Compare All Corrector Sizes

Pick a prompt and see how each corrector size responds.

In [ ]:
# Test prompt
TEST_PROMPT = 'The best way to build AI that you can trust is'
MAX_TOKENS = 60

print(f'Prompt: {TEST_PROMPT}')
print('-'*60)

for size in SIZES_TO_TRAIN:
    if RESULTS.get(size, {}).get('status') != 'success':
        continue
    
    hybrid, tok = HybridModel.load(CIRCUIT_PATH, f'/tmp/corrector_{size}_10k.pt')
    hybrid.corrector.to(device)
    hybrid.corrector.eval()
    
    prompt_ids = tok.encode(TEST_PROMPT)
    reply_ids = generate_reply_hybrid(hybrid, tok, prompt_ids, max_tokens=MAX_TOKENS)
    reply = tok.decode(reply_ids)
    
    r = RESULTS[size]
    print(f'[{size.upper():8} {r["params"]:>8,} params] {reply}')
    print()
    
    del hybrid
    torch.cuda.empty_cache()

## Notes

**Vocabulary size:** 10K BPE vs 4K gives better coverage of names, slang, and rare words in your personal data.

**Scaling law:** Larger correctors generally better at complex reasoning, but diminishing returns kick in around Medium (20M params). Tiny is fast for realtime use.

**Next steps after download:**
1. Download `circuit_pda_10k.json` + the corrector size you want
2. Copy both into your Starfire project
3. Wire into the Rust inference kernel for GPU-free local inference
4. Use evolver to search for better training recipes